# SQL in R Analysis – NorthStar Urban Mobility and Logistics

## Objective
This notebook analyses operational inefficiencies within NorthStar using SQL queries executed within R.

In [1]:
!pip install rpy2

In [2]:
%load_ext rpy2.ipython

In [4]:
!apt-get install -y unixodbc-dev
!R -e "install.packages(c('DBI','RSQLite','dplyr','sqldf'), repos='https://cloud.r-project.org/')"

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unixodbc-dev is already the newest version (2.3.9-5ubuntu0.1).
unixodbc-dev set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.

R version 4.6.0 (2026-04-24) -- "Because it was There"
Copyright (C) 2026 The R Foundation for Statistical Computing
Platform: x86_64-pc-linux-gnu

R is free software and comes with ABSOLUTELY NO WARRANTY.
You are welcome to redistribute it under certain conditions.
Type 'license()' or 'licence()' for distribution details.

  Natural language support but running in an English locale

R is a collaborative project with many contributors.
Type 'contributors()' for more information and
'citation()' on how to cite R or R packages in publications.

Type 'demo()' for some demos, 'help()' for on-line help, or
'help.start()' for an HTML browser interface to help.
Type 'q()' to quit R.

> install.packages(c('DBI','RSQLite','dplyr','sqld

In [5]:
%%R

library(DBI)
library(RSQLite)
library(dplyr)
library(sqldf)


Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union

Loading required package: gsubfn
Loading required package: proto
In addition: Warning message:
no DISPLAY variable so Tk is not available 


In [6]:
from google.colab import files

uploaded = files.upload()

Saving complaints.csv to complaints.csv
Saving deliveries.csv to deliveries.csv
Saving drivers.csv to drivers.csv
Saving hubs.csv to hubs.csv
Saving incidents.csv to incidents.csv
Saving vehicles.csv to vehicles.csv


In [7]:
%%R

deliveries <- read.csv("deliveries.csv")
complaints <- read.csv("complaints.csv")
incidents <- read.csv("incidents.csv")
drivers <- read.csv("drivers.csv")
vehicles <- read.csv("vehicles.csv")
hubs <- read.csv("hubs.csv")

## Dataset Exploration

The deliveries dataset contains structured operational delivery records including delivery status, route override activity, customer ratings, and operational cost information.

In [8]:
%%R

head(deliveries)

  delivery_id order_id driver_id vehicle_id hub_id       dispatch_time
1     DL00001   O00938      D004       V056    H05 2024-06-18 10:57:00
2     DL00002   O00004      D138       V007    H02 2025-01-11 18:45:00
3     DL00003   O00639      D006       V049    H02 2025-06-02 20:39:00
4     DL00004   O00313      D116       V055    H02 2024-03-08 23:31:00
5     DL00005   O00844      D108       V034    H01 2025-09-21 11:43:00
6     DL00006   O00029      D037       V098    H03 2024-09-11 12:40:00
       delivery_completed_at delivery_status route_distance_km
1 2024-06-19 09:05:59.904311          Failed             17.26
2 2025-01-11 17:39:00.000000          OnTime             10.34
3 2025-06-02 21:45:32.366770          OnTime              7.92
4 2024-03-09 23:30:08.103702         Delayed             16.42
5 2025-09-21 15:45:34.131056          OnTime             14.52
6 2024-09-12 17:11:52.384869         Delayed             13.84
  manual_route_override_count proof_of_completion_missing
1   

In [10]:
%%R

colnames(deliveries)

 [1] "delivery_id"                   "order_id"                     
 [3] "driver_id"                     "vehicle_id"                   
 [5] "hub_id"                        "dispatch_time"                
 [7] "delivery_completed_at"         "delivery_status"              
 [9] "route_distance_km"             "manual_route_override_count"  
[11] "proof_of_completion_missing"   "customer_rating_post_delivery"
[13] "fuel_or_charge_cost"          


## Query 1 – Failed Deliveries and Operational Performance by Hub

### Objective
Identify operational hubs with elevated failed delivery frequencies, increased route override activity, and reduced customer satisfaction ratings.

In [11]:
%%R

query1 <- "
SELECT hub_id,
       COUNT(*) AS failed_deliveries,
       AVG(manual_route_override_count) AS avg_route_overrides,
       AVG(customer_rating_post_delivery) AS avg_customer_rating
FROM deliveries
WHERE delivery_status = 'Failed'
GROUP BY hub_id
ORDER BY failed_deliveries DESC;
"

failed_hubs <- sqldf(query1)

print(failed_hubs)

  hub_id failed_deliveries avg_route_overrides avg_customer_rating
1    H08                26           1.1153846            3.193462
2    H05                23           1.0434783            2.929091
3    H01                17           1.1176471            2.786471
4    H04                16           1.0000000            2.998125
5    H06                15           0.8666667            3.069333
6    H07                14           1.0714286            3.334286
7    H03                11           1.0909091            2.910909
8    H02                10           0.9000000            3.191000


## Interpretation

Several operational hubs demonstrated elevated failed delivery frequencies combined with increased route override activity and reduced customer ratings.

This suggests that operational instability and route-management inefficiencies may contribute directly to declining service quality.

In [13]:
%%R

colnames(incidents)

[1] "incident_id"       "delivery_id"       "incident_type"    
[4] "reported_at"       "severity"          "resolution_status"
[7] "resolved_hours"   


## Query 2 – Operational Incident Severity Analysis

### Objective
Identify the most common operational incident types and evaluate incident resolution performance.

In [14]:
%%R

query2 <- "
SELECT incident_type,
       COUNT(*) AS incident_count,
       AVG(resolved_hours) AS avg_resolution_time
FROM incidents
GROUP BY incident_type
ORDER BY incident_count DESC;
"

incident_analysis <- sqldf(query2)

print(incident_analysis)

     incident_type incident_count avg_resolution_time
1     ProofMissing             46           10.767500
2   CustomerNoShow             44           13.888095
3   RouteDeviation             43           13.726829
4     VehicleFault             37            9.150000
5     BatteryAlert             36           11.708824
6     AppSyncError             31           12.657143
7 TemperatureIssue             29           12.917241
8   SafetyNearMiss             14            9.669231


## Interpretation

Several operational incident categories demonstrated elevated occurrence frequencies and extended resolution times.

This suggests that certain operational failures may contribute disproportionately to delivery disruption and service instability.

## Query 3 – Vehicle Incident Frequency Analysis

### Objective
Identify vehicles associated with elevated operational incident frequencies.

In [16]:
%%R

query3 <- "
SELECT d.vehicle_id,
       COUNT(i.incident_id) AS incident_count
FROM deliveries d
JOIN incidents i
ON d.delivery_id = i.delivery_id
GROUP BY d.vehicle_id
ORDER BY incident_count DESC;
"

vehicle_incidents <- sqldf(query3)

print(head(vehicle_incidents, 10))

   vehicle_id incident_count
1        V047              9
2        V108              7
3        V097              6
4        V046              6
5        V030              6
6        V088              5
7        V076              5
8        V042              5
9        V035              5
10       V009              5


## Interpretation

Several vehicles were associated with elevated operational incident frequencies, suggesting that vehicle condition and maintenance performance may contribute directly to operational instability and service disruption.

In [17]:
%%R

colnames(complaints)

 [1] "complaint_id"        "customer_id"         "order_id"           
 [4] "complaint_type"      "channel"             "severity"           
 [7] "created_at"          "status"              "resolution_days"    
[10] "compensation_amount"


## Query 4 – Complaint Severity and Compensation Analysis

### Objective
Analyse complaint severity levels and compensation costs across operational service failures.

In [18]:
%%R

query4 <- "
SELECT complaint_type,
       severity,
       COUNT(*) AS complaint_count,
       AVG(compensation_amount) AS avg_compensation
FROM complaints
GROUP BY complaint_type, severity
ORDER BY avg_compensation DESC;
"

complaint_analysis <- sqldf(query4)

print(head(complaint_analysis, 10))

      complaint_type severity complaint_count avg_compensation
1            Billing     High               4         52.32250
2       MissedPickup     High              16         43.06938
3    DriverBehaviour     High              16         38.38583
4             Damage     High               7         37.26429
5              Delay     High              18         36.53857
6           AppIssue     High              13         34.04917
7  SupportExperience     High               3         33.88333
8  SupportExperience   Medium              12         18.67750
9              Delay   Medium              56         18.20509
10      MissedPickup   Medium              37         17.91111


## Interpretation

Several complaint categories demonstrated elevated severity levels and higher compensation costs.

This suggests that operational service failures contribute directly to customer dissatisfaction and increased financial losses.

# Final SQL Analysis Discussion

The SQL in R analysis identified several operational inefficiencies across NorthStar’s structured operational systems.

Key findings include:

- operational hubs with elevated failed delivery frequencies
- increased route override activity
- repeated operational incident patterns
- vehicle-related operational instability
- strong relationships between complaint severity and compensation costs

The analysis demonstrates that customer dissatisfaction and operational instability are interconnected across multiple business functions.

These findings support the argument that NorthStar’s fragmented systems reduce operational visibility and limit effective decision-making.